# Suite CPP — The Ccpp language

C and C++ as one tree language (`mbse.Programs.Ccpp.Syntax`): its kinds and categories, organized as the standard's
grammar is, the identifiers it accepts, and where each kind and feature exists, which the C++ standards check.

In [ ]:
from mbse.Programs.Ccpp import Ccpp17, Ccpp20, Syntax as S
from mbse.Programs.Ccpp._Standard import CcppStandard
from mbse.Programs.Framework import Syntax as F

L = S.LANGUAGE
C23, CPP23, CPP26 = CcppStandard(2023, "C"), CcppStandard(2023), CcppStandard(2026)


def name(spelling):
    return S.Identifier(spelling=spelling)


def ident(spelling):
    return S.IdExpression(name=name(spelling))


def one():
    return S.IntegerLiteral(spelling="1")

## CPP-01 · Every kind belongs to one category, and the grammar lists them

In [ ]:
kinds = L.kinds()
assert len(kinds) == len(S.KINDS) and list(kinds) == [k.__name__ for k in S.KINDS]
assert list(L.categories()) == ["Name", "Expression", "Literal", "Capture", "Requirement", "Designator", "Statement",
                                "Declaration", "Specifier", "Declarator", "Parameter", "Initializer",
                                "ExceptionSpecification", "ContractSpecifier", "AttributeSpecifier",
                                "TemplateParameter", "Directive"]
assert issubclass(S.IntegerLiteral, S.Literal) and issubclass(S.Literal, S.Expression)
assert all(k.NAME == f"Programs.Ccpp.{k.KIND}" for k in S.KINDS)
grammar = L.grammar()
assert grammar["base"] == {"C++": 1998, "C": 1989} and len(grammar["kinds"]) == len(S.KINDS)
assert {"name": "Literal", "base": "Expression"} in grammar["categories"]
assert S.cpp(2011) == {"C++": 2011} and S.c(1999) == {"C": 1999} and S.both(2011, 2023) == {"C++": 2011, "C": 2023}
assert S.Comment(text=" x").block is False and S.TranslationUnit().items == []
print("ok")

## CPP-02 · Identifiers are letters, digits, `_` and `$`, not starting with a digit

In [ ]:
assert L.validate(name("x_1$")) == [] and L.validate(name("Größe")) == []
assert L.validate(name("1x")) == ["'1x' is not an identifier"]
assert L.validate(name("a b")) == ["'a b' is not an identifier"]
assert L.validate(name("")) == [] and L.validate(S.Identifier()) == ["an Identifier needs a spelling"]
assert L.validate(S.Identifier(spelling=3)) == ["Identifier.spelling must be a str, got int"]
print("ok")

## CPP-03 · Where kinds exist: C++ only, C only, both, and since which standard

In [ ]:
lambda_ = S.LambdaExpression(body=S.CompoundStatement())
assert Ccpp17.check(lambda_) == [] and C23.check(lambda_) == ["LambdaExpression is not C"]
generic = S.GenericSelection(controlling=one(), associations=[S.GenericAssociation(value=one())])
assert C23.check(generic) == []
assert Ccpp20.check(generic) == ["GenericSelection is not C++", "associations[0]: GenericAssociation is not C++"]
requires = S.RequiresExpression(requirements=[S.SimpleRequirement(expression=one())])
assert Ccpp17.check(requires) == ["RequiresExpression needs C++20", "requirements[0]: SimpleRequirement needs C++20"]
assert Ccpp20.check(requires) == []
throw = S.ThrowSpecifier(types=[S.TypeId(specifiers=[S.PrimitiveTypeSpecifier(keyword="int")])])
assert Ccpp17.check(throw) == ["ThrowSpecifier with types was removed in C++17"]
assert Ccpp17.check(S.ThrowSpecifier()) == [] and Ccpp20.check(S.ThrowSpecifier()) == ["ThrowSpecifier was removed in C++20"]
assert Ccpp17.check(S.DeclSpecifier(keyword="register")) == ["DeclSpecifier.keyword 'register' was removed in C++17"]
assert C23.check(S.DeclSpecifier(keyword="register")) == [] and Ccpp20.check(S.DeclSpecifier(keyword="__inline")) == []
assert Ccpp20.check(S.StatementExpression(body=S.CompoundStatement())) == []  # an extension
assert Ccpp17.check(S.BinaryExpression(left=one(), operator="<=>", right=one())) == [
    "BinaryExpression.operator '<=>' needs C++20"]
assert Ccpp17.check(S.IfStatement(constexpr=True, consequence=S.CompoundStatement())) == []
assert Ccpp20.check(S.IfStatement(consteval=True, consequence=S.CompoundStatement())) == [
    "IfStatement.consteval needs C++23"]
print("ok")

## CPP-04 · Kinds whose features depend on their fields

In [ ]:
subscript = S.SubscriptExpression(object=ident("m"), indices=[one(), one()])
assert Ccpp20.check(subscript) == ["SubscriptExpression with several indices needs C++23"]
assert CPP23.check(subscript) == [] and Ccpp20.check(S.SubscriptExpression(object=ident("m"), indices=[one()])) == []
field = S.FieldDesignator(name=name("x"))
designated = S.DesignatedInitializer(designators=[field, S.IndexDesignator(index=one())],
                                     initializer=S.EqualInitializer(value=one()))
assert Ccpp20.check(designated) == ["DesignatedInitializer with array or nested designators is not C++",
                                    "designators[1]: IndexDesignator is not C++"]
assert C23.check(designated) == [] and Ccpp17.check(S.DesignatedInitializer(
    designators=[field], initializer=S.EqualInitializer(value=one()))) == ["DesignatedInitializer needs C++20"]
for label in (S.LabeledStatement(label=name("l")), S.CaseStatement(value=one()), S.DefaultStatement()):
    assert Ccpp20.check(label) == [f"{label.KIND} without a statement needs C++23"]
    label.statement = S.BreakStatement()
    assert Ccpp20.check(label) == []
assert C23.check(S.TypeofSpecifier(keyword="typeof", operand=one())) == []
assert Ccpp20.check(S.TypeofSpecifier(keyword="typeof", operand=one())) == ["TypeofSpecifier is not C++"]
assert Ccpp20.check(S.TypeofSpecifier(keyword="__typeof__", operand=one())) == []
bindings = S.StructuredBindingDeclarator(bindings=[S.IdDeclarator(name=name("a"))])
assert Ccpp17.check(bindings) == []
bindings.bindings += [S.PackDeclarator(declarator=S.IdDeclarator(name=name("r"))),
                      S.IdDeclarator(name=name("b"), attributes=[S.StandardAttributeSpecifier()])]
assert Ccpp20.check(bindings) == ["StructuredBindingDeclarator with a pack needs C++26",
                                  "StructuredBindingDeclarator with attributes needs C++26"]
assert CPP26.check(bindings) == []
nested = S.NamespaceDefinition(names=[S.NamespaceName(name=name("a")), S.NamespaceName(name=name("b"))])
assert Ccpp17.check(nested) == [] and Ccpp17.check(S.NamespaceDefinition(
    names=[S.NamespaceName(name=name("a"))])) == []
nested.names[1].inline = True
assert Ccpp17.check(nested) == ["NamespaceDefinition with inline nested names needs C++20"]
assert CcppStandard(2014).check(nested)[0] == "NamespaceDefinition with nested names needs C++17"
using = S.UsingDeclaration(declarators=[S.UsingDeclarator(name=name("a"))] * 2)
assert CcppStandard(2014).check(using) == ["UsingDeclaration with several declarators needs C++17"]
assert Ccpp17.check(using) == []
assertion = S.StaticAssertDeclaration(keyword="static_assert", condition=one())
assert CcppStandard(2014).check(assertion) == ["StaticAssertDeclaration without a message needs C++17"]
assertion.message = S.StringLiteral(text="m")
assert CcppStandard(2014).check(assertion) == []
assert Ccpp20.check(S.OtherDirective(directive="warning")) == ["#warning needs C++23"]
assert Ccpp20.check(S.OtherDirective(directive="embed")) == ["#embed needs C++26"]
assert Ccpp17.check(S.OtherDirective(directive="pragma")) == [] and Ccpp17.check(S.OtherDirective()) == []
print("ok")